# Bookmaker Historical Backfill + Multiplicative De-vig Calibration

- Owner: 老彭 (betting-industry-expert, C 单元 IC)
- Last review: 2026-05-28
- Input: Goalserve 历史 odds (CSV / Parquet, 小余 / 小段 提供)
- Output: 偏差统计表 + 异常案例 + Shin 触发判断
- 关联: laopeng-bookmaker-history-backfill-v1.md / laopeng-multiplicative-devig-calibration-v1.md
- ML-R2: Python offline notebook OK (不进生产 cpp)
- 数据范围: 2024-05 ~ 2026-05, 8 sport, 8-9 家 bookmaker

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 数据目录 (小余 / 小段 提供 Parquet 后改这里)
DATA_DIR = Path('../../docs/RESEARCH/data')
ODDS_FILE = DATA_DIR / 'goalserve_odds_history.parquet'  # 待小余提供
SETTLEMENT_FILE = DATA_DIR / 'goalserve_settlement_history.parquet'  # 待小余提供

# 如果没有真实数据，使用 stub
USE_STUB = not ODDS_FILE.exists()
print(f'Data mode: {"STUB" if USE_STUB else "REAL"}')
print(f'Odds file: {ODDS_FILE}')

## 1. 数据加载 + 基础清洗

In [ ]:
# --- Schema 定义 (与 laopeng-bookmaker-history-backfill-v1.md §4.1 对齐) ---
SCHEMA_DTYPES = {
    'event_ts': 'int64',          # 比赛开始时间 UTC ms (R-20 ts-1)
    'data_source_ts': 'int64',    # Goalserve @ts unix ms (R-20 ts-2)
    'ingestion_ts': 'int64',      # ETL 拉取时间 ms (R-20 ts-3)
    'as_of_ts': 'int64',          # 快照时间点 ms (R-20 ts-4)
    'sport': 'str',
    'league_id': 'str',
    'match_id': 'str',
    'bookmaker_id': 'int32',
    'bookmaker_name': 'str',
    'market_id': 'str',
    'market_type': 'str',         # moneyline / totals / spreads
    'outcome_name': 'str',        # Home / Away / Draw / Over / Under
    'handicap': 'float64',
    'odds_eu': 'float64',
    'is_suspended': 'bool',
    'settlement_result': 'str',   # Win / Loose / Stake_refund / Half_win / Half_loose
}

SPORTS = ['soccer', 'basket', 'tennis', 'baseball', 'amfootball', 'hockey', 'volleyball', 'esports']
MARKET_TYPES = ['moneyline', 'totals', 'spreads']
BOOKMAKER_IDS = [14, 15, 16, 17, 18, 65, 105, 144]  # 10Bet/WH/bet365/Marathon/Unibet/BetVictor/1xBet/Betano

# 假球黑名单 (laopeng-bookmaker-history-backfill-v1.md §5)
BLACKLIST_LEAGUES = [
    'ITF M15', 'ITF W15', 'ITF M25', 'ITF W25',
    'Setka Cup', 'eSoccer', 'virtual'
]

print('Schema defined.')
print(f'Sports: {SPORTS}')
print(f'Bookmakers: {BOOKMAKER_IDS}')

In [ ]:
if USE_STUB:
    # --- STUB DATA for development / dry run ---
    # 生成 synthetic odds data 用于验证 pipeline 逻辑
    # 真实数据由小余 / 小段 提供 Parquet 后替换
    np.random.seed(42)
    N_EVENTS = 5000  # stub 用 5000 场模拟 (真实 ~68500)
    N_BM = 8

    records = []
    for event_i in range(N_EVENTS):
        sport = np.random.choice(SPORTS)
        market_type = np.random.choice(MARKET_TYPES, p=[0.5, 0.3, 0.2])
        event_ts = int(pd.Timestamp('2024-05-01').timestamp() * 1000) + \
                   event_i * 3600 * 1000 * 8

        # 生成 true prob (合成)
        if market_type == 'moneyline':
            outcomes = ['Home', 'Away', 'Draw'] if sport == 'soccer' else ['Home', 'Away']
            true_probs = np.random.dirichlet(np.ones(len(outcomes)))
        else:
            outcomes = ['Over', 'Under']
            true_probs = np.array([0.5 + np.random.normal(0, 0.05),
                                   0.5 - np.random.normal(0, 0.05)])
            true_probs = np.clip(true_probs, 0.1, 0.9)
            true_probs /= true_probs.sum()

        # 决定 actual outcome
        actual_outcome = np.random.choice(outcomes, p=true_probs)

        for bm_id in BOOKMAKER_IDS:
            # 各家 bookmaker 加 vig + 噪声
            vig_rate = np.random.uniform(0.03, 0.07)
            for i, outcome in enumerate(outcomes):
                # implied prob = true_prob / (1 - vig_rate) + noise
                implied = true_probs[i] * (1 + vig_rate) + np.random.normal(0, 0.005)
                implied = max(implied, 0.01)
                odds_eu = round(1.0 / implied, 2)
                records.append({
                    'event_ts': event_ts,
                    'data_source_ts': event_ts - 3600 * 1000,
                    'ingestion_ts': event_ts - 3000 * 1000,
                    'as_of_ts': event_ts - 3600 * 1000,
                    'sport': sport,
                    'league_id': f'league_{sport[:3]}_{np.random.randint(1,10)}',
                    'match_id': f'match_{event_i:06d}',
                    'bookmaker_id': bm_id,
                    'bookmaker_name': {14:'10bet',15:'williamhill',16:'bet365',
                                       17:'marathon',18:'unibet',65:'betvictor',
                                       105:'1xbet',144:'betano'}[bm_id],
                    'market_id': '1' if market_type == 'moneyline' else '421',
                    'market_type': market_type,
                    'outcome_name': outcome,
                    'handicap': 0.0,
                    'odds_eu': odds_eu,
                    'is_suspended': False,
                    'settlement_result': 'Win' if outcome == actual_outcome else 'Loose',
                })

    df_raw = pd.DataFrame(records)
    print(f'STUB data: {len(df_raw):,} rows, {df_raw["match_id"].nunique():,} events')
else:
    df_raw = pd.read_parquet(ODDS_FILE)
    print(f'Real data: {len(df_raw):,} rows, {df_raw["match_id"].nunique():,} events')

df_raw.head(3)

In [ ]:
# 数据清洗
df = df_raw.copy()

# 1. 过滤 suspended odds
df = df[~df['is_suspended']]

# 2. 过滤 odds 合理范围 (laopeng-backfill §5)
df = df[(df['odds_eu'] >= 1.01) & (df['odds_eu'] <= 50.0)]

# 3. 过滤假球黑名单联赛
df = df[~df['league_id'].str.contains('|'.join(BLACKLIST_LEAGUES), case=False, na=False)]

# 4. 只保留已结算的 events (settlement_result 非空)
df = df[df['settlement_result'].notna() & (df['settlement_result'] != '')]

# 5. R-20 时间戳顺序检查
ts_valid = (df['event_ts'] >= df['data_source_ts']) | (df['data_source_ts'] <= df['ingestion_ts'])
df = df[df['data_source_ts'] <= df['ingestion_ts']]

print(f'After cleaning: {len(df):,} rows, {df["match_id"].nunique():,} events')
print(f'Sport distribution:')
print(df['sport'].value_counts())

## 2. Multiplicative De-vig 计算

In [ ]:
def multiplicative_devig_single_book(odds_list):
    """
    对单家 bookmaker 的一组 outcome 欧赔做 multiplicative de-vig.
    returns: fair probabilities (list, same order as odds_list)
    laopeng-multiplicative-devig-calibration-v1.md §1.3
    """
    implied = [1.0 / o for o in odds_list]
    total = sum(implied)
    if total <= 0:
        return [1.0 / len(odds_list)] * len(odds_list)  # fallback: uniform
    return [p / total for p in implied]


def compute_fair_value_per_event_market(group):
    """
    对一个 (match_id, market_type) 分组，计算每个 outcome 的 fair_value.
    等权跨家均值 (ADR-008 核心).
    """
    outcomes = group['outcome_name'].unique()
    bookmakers = group['bookmaker_id'].unique()

    # 每家 bookmaker 做一次 multiplicative de-vig
    bm_fair_probs = {}  # {bm_id: {outcome: fair_prob}}
    for bm_id in bookmakers:
        bm_data = group[group['bookmaker_id'] == bm_id]
        bm_outcomes = bm_data['outcome_name'].tolist()
        bm_odds = bm_data['odds_eu'].tolist()
        if len(bm_odds) < 2:
            continue
        fair_probs = multiplicative_devig_single_book(bm_odds)
        bm_fair_probs[bm_id] = dict(zip(bm_outcomes, fair_probs))

    if len(bm_fair_probs) < 1:
        return pd.DataFrame()

    # 等权跨家均值
    results = []
    for outcome in outcomes:
        probs = [bm_fair_probs[b][outcome]
                 for b in bm_fair_probs if outcome in bm_fair_probs[b]]
        if len(probs) == 0:
            continue
        fair_value = np.mean(probs)
        n_bookmakers = len(probs)

        # actual outcome (settlement)
        outcome_rows = group[group['outcome_name'] == outcome]
        settlement = outcome_rows['settlement_result'].iloc[0] if len(outcome_rows) > 0 else None
        actual = 1.0 if settlement == 'Win' else (0.5 if settlement in ['Half_win', 'Half_loose'] else 0.0)

        results.append({
            'outcome_name': outcome,
            'fair_value': fair_value,
            'actual': actual,
            'bias': actual - fair_value,
            'n_bookmakers': n_bookmakers,
            'low_coverage': n_bookmakers < 6,
        })

    return pd.DataFrame(results)


print('De-vig functions defined.')

In [ ]:
# 按 match_id × market_type 分组计算 fair_value
# 对 stub 5000 events × 8 bm = 40000 rows，这里合理快速

devig_results = []

# 添加 event metadata (取第一行)
event_meta = df.groupby('match_id').agg(
    sport=('sport', 'first'),
    event_ts=('event_ts', 'first'),
    league_id=('league_id', 'first'),
).reset_index()

for (match_id, market_type), group in df.groupby(['match_id', 'market_type']):
    result_df = compute_fair_value_per_event_market(group)
    if result_df.empty:
        continue
    result_df['match_id'] = match_id
    result_df['market_type'] = market_type
    devig_results.append(result_df)

df_devig = pd.concat(devig_results, ignore_index=True)

# 合并 event metadata
df_devig = df_devig.merge(event_meta, on='match_id', how='left')

# 添加 week 字段 (for P1 trigger check)
df_devig['event_dt'] = pd.to_datetime(df_devig['event_ts'], unit='ms', utc=True)
df_devig['week'] = df_devig['event_dt'].dt.to_period('W')

print(f'De-vig computed: {len(df_devig):,} outcome rows, {df_devig["match_id"].nunique():,} events')
print(f'Columns: {list(df_devig.columns)}')
df_devig.head(5)

## 3. 偏差统计 (per sport × market_type)

In [ ]:
# 过滤 low_coverage (< 6 家 bookmaker)
df_valid = df_devig[~df_devig['low_coverage']].copy()
df_low_cov = df_devig[df_devig['low_coverage']]
print(f'Valid rows: {len(df_valid):,}, Low coverage excluded: {len(df_low_cov):,}')

# Per sport × market_type 偏差统计
bias_stats = df_valid.groupby(['sport', 'market_type']).agg(
    n_outcomes=('bias', 'count'),
    mean_bias=('bias', 'mean'),
    std_bias=('bias', 'std'),
    abs_bias_gt_001=('bias', lambda x: (x.abs() > 0.01).mean()),
    p5_bias=('bias', lambda x: x.quantile(0.05)),
    p95_bias=('bias', lambda x: x.quantile(0.95)),
).reset_index()

bias_stats.columns = ['sport', 'market_type', 'n_outcomes', 'mean_bias', 'std_bias',
                       '|bias|>0.01_rate', 'p5_bias', 'p95_bias']

# 格式化输出
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 40)
print('\n=== Bias Statistics: Per Sport × Market Type ===')
print(bias_stats.to_string(index=False))

In [ ]:
# 小梁 P1 触发检测: 任意盘口 |bias| > 0.01 连续 3 周
print('\n=== P1 Trigger Detection: |bias| > 0.01 consecutive 3 weeks ===')

weekly_bias = df_valid.groupby(['sport', 'market_type', 'week']).agg(
    mean_abs_bias=('bias', lambda x: x.abs().mean()),
    n_outcomes=('bias', 'count'),
).reset_index()

weekly_bias['trigger'] = weekly_bias['mean_abs_bias'] > 0.01

shin_candidates = []
for (sport, market_type), group in weekly_bias.groupby(['sport', 'market_type']):
    group_sorted = group.sort_values('week')
    triggers = group_sorted['trigger'].tolist()
    # check 3 consecutive
    consecutive_3 = False
    max_consecutive = 0
    cur = 0
    for t in triggers:
        if t:
            cur += 1
            max_consecutive = max(max_consecutive, cur)
            if cur >= 3:
                consecutive_3 = True
        else:
            cur = 0

    if consecutive_3:
        shin_candidates.append({
            'sport': sport,
            'market_type': market_type,
            'max_consecutive_weeks': max_consecutive,
            'shin_trigger': True,
        })

if shin_candidates:
    df_shin = pd.DataFrame(shin_candidates)
    print('SHIN UPGRADE CANDIDATES:')
    print(df_shin.to_string(index=False))
else:
    print('No Shin trigger detected. Multiplicative de-vig calibration OK.')

## 4. Totals 对称性检验

In [ ]:
# Totals 盘口对称假设: mean(fair_value(Over)) ≈ 0.5
# laopeng-multiplicative-devig-calibration-v1.md §4

df_totals = df_valid[
    (df_valid['market_type'] == 'totals') &
    (df_valid['outcome_name'] == 'Over')
].copy()

if len(df_totals) > 0:
    print('=== Totals Symmetry Check: fair_value(Over) distribution ===')
    totals_sym = df_totals.groupby('sport').agg(
        n=('fair_value', 'count'),
        mean_fv_over=('fair_value', 'mean'),
        std_fv_over=('fair_value', 'std'),
        mean_bias_over=('bias', 'mean'),
    ).reset_index()
    totals_sym['asymmetry_from_0.5'] = totals_sym['mean_fv_over'] - 0.5

    # t-test H0: mean_fv_over = 0.5
    for sport in totals_sym['sport']:
        sport_data = df_totals[df_totals['sport'] == sport]['fair_value']
        if len(sport_data) >= 30:
            t_stat, p_val = stats.ttest_1samp(sport_data, 0.5)
            totals_sym.loc[totals_sym['sport'] == sport, 't_stat'] = round(t_stat, 3)
            totals_sym.loc[totals_sym['sport'] == sport, 'p_value'] = round(p_val, 4)
            totals_sym.loc[totals_sym['sport'] == sport, 'symmetric_H0'] = 'REJECT' if p_val < 0.05 else 'ACCEPT'

    print(totals_sym.to_string(index=False))
    print('\n老彭注: 如果 p_value < 0.05 且 asymmetry_from_0.5 > 0.01, 说明 Over 系统性高估 → square book Under 有 edge')
else:
    print('No totals data available.')

## 5. 异常案例分析 (Top 10 |bias|)

In [ ]:
# Top 10 最大偏差案例
# laopeng-multiplicative-devig-calibration-v1.md §5

df_devig['abs_bias'] = df_devig['bias'].abs()

top_anomalies = (
    df_devig
    .nlargest(10, 'abs_bias')
    [['event_dt', 'sport', 'league_id', 'match_id', 'market_type',
      'outcome_name', 'fair_value', 'actual', 'bias', 'abs_bias', 'n_bookmakers', 'low_coverage']]
    .round(4)
)

print('=== Top 10 Anomaly Cases (|bias| largest) ===')
print(top_anomalies.to_string(index=False))

# 异常成因分类 (自动标注)
def classify_anomaly(row):
    if row['low_coverage']:
        return 'low_bm_coverage'
    if any(bl in str(row['league_id']) for bl in BLACKLIST_LEAGUES):
        return 'blacklist_league'
    if row['abs_bias'] > 0.3:
        return 'extreme_outlier_check_manually'
    if row['abs_bias'] > 0.10:
        return 'large_bias_shin_candidate'
    return 'normal_variance'

top_anomalies['anomaly_type'] = top_anomalies.apply(classify_anomaly, axis=1)
print('\n=== Anomaly Classification ===')
print(top_anomalies[['match_id', 'sport', 'market_type', 'abs_bias', 'anomaly_type']].to_string(index=False))

## 6. Alpha 估计 (hit rate + Sharpe prior)

In [ ]:
# P0-01 GoalserveDevig 信号 hit rate 估计
# 策略: 买 fair_value > 0.55 的 outcome (明显被市场低估)
# laopeng-multiplicative-devig-calibration-v1.md §7

EDGE_THRESHOLD = 0.02  # 进场阈值: fair_value > 0.5 + threshold

df_signal = df_valid[
    (df_valid['fair_value'] > 0.5 + EDGE_THRESHOLD) &
    (~df_valid['low_coverage'])
].copy()

print(f'Signal candidates (fair_value > {0.5 + EDGE_THRESHOLD:.2f}): {len(df_signal):,}')

# Hit rate by sport × market_type
hit_rate_table = df_signal.groupby(['sport', 'market_type']).agg(
    n=('actual', 'count'),
    hit_rate=('actual', 'mean'),
    mean_fair_value=('fair_value', 'mean'),
    mean_edge=('bias', 'mean'),
).reset_index()
hit_rate_table['beat_random'] = hit_rate_table['hit_rate'] - 0.5

print('\n=== Hit Rate Table (P0-01 signal proxy) ===')
print(hit_rate_table.to_string(index=False))

# 全局汇总
global_hit = df_signal['actual'].mean()
global_std = df_signal['actual'].std()
print(f'\nGlobal hit rate: {global_hit:.4f} ({global_hit*100:.2f}%)')
print(f'小梁预设区间: 53-55%')
print(f'老彭校准区间: 54-57% (Moneyline), 52-55% (Totals)')
if 0.53 <= global_hit <= 0.58:
    print('RESULT: Within expected range.')
elif global_hit > 0.58:
    print('RESULT: Above expected range. Check for data leakage or stub artifact.')
else:
    print('RESULT: Below expected range. Review de-vig methodology or data quality.')

In [ ]:
# 简单 Sharpe 估计 (单边，不含 Polymarket spread)
# 假设: 每注 $1, 赔率取 fair_value 倒数 (no-vig)

df_signal['pnl'] = np.where(
    df_signal['actual'] == 1.0,
    (1.0 / df_signal['fair_value']) - 1,   # win: net profit
    -1.0                                    # loss: stake lost
)

mean_pnl = df_signal['pnl'].mean()
std_pnl = df_signal['pnl'].std()

# 按周聚合 Sharpe
weekly_pnl = df_signal.groupby('week')['pnl'].mean()
sharpe_weekly = (weekly_pnl.mean() / weekly_pnl.std()) * np.sqrt(52)  # annualized

print(f'=== Sharpe Estimate ===')
print(f'Mean PnL per bet: {mean_pnl:.4f} ({mean_pnl*100:.2f}%)')
print(f'Std PnL per bet: {std_pnl:.4f}')
print(f'Annualized Sharpe (weekly): {sharpe_weekly:.2f}')
print(f'小梁北极星: Sharpe ≥ 1.5')
print(f'老彭校准区间: 1.2-1.8')

print(f'\n给小董 M4.5 G7 random baseline:')
print(f'  Random (50%): Sharpe ≈ 0.0')
print(f'  Square money (48%): Sharpe ≈ -0.3')
print(f'  老彭 prior (multiplicative de-vig): Sharpe = {sharpe_weekly:.2f}')

## 7. 输出汇总 + 跨单元通知

In [ ]:
print('='*60)
print('SUMMARY: Multiplicative De-vig Calibration v1')
print('='*60)
print(f'Data: {"STUB (待真实数据替换)" if USE_STUB else "REAL"}')
print(f'Events: {df_devig["match_id"].nunique():,}')
print(f'Sports: {df_devig["sport"].nunique()} / 8 target')
print(f'Weeks covered: {df_devig["week"].nunique()}')
print()
print('Key findings:')
print(f'  - Global hit rate: {global_hit*100:.2f}% (target: 53-57%)')
print(f'  - Annualized Sharpe: {sharpe_weekly:.2f} (target: ≥ 1.5)')
print(f'  - Shin trigger candidates: {len(shin_candidates)}')
if shin_candidates:
    for c in shin_candidates:
        print(f'    - {c["sport"]} × {c["market_type"]}: {c["max_consecutive_weeks"]} consecutive weeks')
print()
print('Cross-unit notifications:')
print('  @小梁: P1 触发判断见上方 Shin trigger section。W6-W7 EOD 前完成 ack。')
print('  @小余: ETL schema 见 laopeng-bookmaker-history-backfill-v1.md §4.1')
print('  @小段: Goalserve 历史 API 覆盖范围确认，2024-05 起是否可拉')
print('  @小董: M4.5 G7 random baseline 数据见 §7.2 校准文档')
print('  @小邓: settlement_result 字段可直接用作 ML training label')
print('  @小卢: ADR-008 cpp 实现校准结果反馈，fair_value 偏差均值供 cpp 单元测试 sanity check')